In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import to_timestamp, hour, when, col, lit, count

In [2]:
spark = SparkSession.builder.appName("W4GA_DF").getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/10/19 17:14:01 INFO SparkEnv: Registering MapOutputTracker
25/10/19 17:14:01 INFO SparkEnv: Registering BlockManagerMaster
25/10/19 17:14:01 INFO SparkEnv: Registering BlockManagerMasterHeartbeat
25/10/19 17:14:01 INFO SparkEnv: Registering OutputCommitCoordinator


In [3]:
import os
_bucket = os.environ.get("GCS_BUCKET", "gs://your-bucket-name").rstrip("/")
input_path = f"{_bucket}/w4/click_file.txt"
output_path = f"{_bucket}/w4/w4_df_output"

In [4]:
df = spark.read.option("header", "true").csv(input_path)
df.show()

+-------+-------------------+
|user_id|          timestamp|
+-------+-------------------+
|   u001|2025-08-01 09:03:23|
|   u012|2025-08-01 12:37:50|
|   u006|2025-08-01 06:57:32|
|   u006|2025-08-01 21:09:33|
|   u029|2025-08-01 14:00:44|
|   u027|2025-08-01 00:52:08|
|   u044|2025-08-01 00:46:59|
|   u028|2025-08-01 20:07:15|
|   u001|2025-08-01 21:21:52|
|   u034|2025-08-01 08:47:16|
|   u023|2025-08-01 06:36:25|
|   u028|2025-08-01 18:46:47|
|   u047|2025-08-01 11:21:19|
|   u031|2025-08-01 07:41:13|
|   u043|2025-08-01 13:40:18|
|   u003|2025-08-01 02:06:14|
|   u010|2025-08-01 08:48:40|
|   u037|2025-08-01 08:17:49|
|   u016|2025-08-01 20:00:19|
|   u030|2025-08-01 05:51:50|
+-------+-------------------+
only showing top 20 rows



In [5]:
df2 = df.withColumn("ts", to_timestamp(col("timestamp"), "yyyy-MM-dd HH:mm:ss")) \
        .withColumn("hour", hour(col("ts")))
df2.head()

Row(user_id='u001', timestamp='2025-08-01 09:03:23', ts=datetime.datetime(2025, 8, 1, 9, 3, 23), hour=9)

In [6]:
bin_col = when((col("hour") >= 0) & (col("hour") < 6), "0-6") \
    .when((col("hour") >= 6) & (col("hour") < 12), "6-12") \
    .when((col("hour") >= 12) & (col("hour") < 18), "12-18") \
    .when((col("hour") >= 18) & (col("hour") < 24), "18-24") \
    .otherwise("unknown")

In [7]:
df3 = df2.withColumn("bin", bin_col)
df3.head()

Row(user_id='u001', timestamp='2025-08-01 09:03:23', ts=datetime.datetime(2025, 8, 1, 9, 3, 23), hour=9, bin='6-12')

In [8]:
counts = df3.groupBy("bin").agg(count(lit(1)).alias("click_count"))
counts.show()

+-----+-----------+
|  bin|click_count|
+-----+-----------+
|12-18|         48|
| 6-12|         53|
|  0-6|         44|
|18-24|         55|
+-----+-----------+



In [9]:
bins_df = spark.createDataFrame(
    [("0-6",), ("6-12",), ("12-18",), ("18-24",)],
    ["bin"]
)
result = bins_df.join(counts, on="bin", how="left").na.fill({"click_count": 0})

In [10]:
print("Click counts by time bin:")
result.show(truncate=False)

Click counts by time bin:


+-----+-----------+
|bin  |click_count|
+-----+-----------+
|0-6  |44         |
|6-12 |53         |
|12-18|48         |
|18-24|55         |
+-----+-----------+



In [11]:
result.coalesce(1).write.mode("overwrite").option("header", "true").csv(output_path)

In [ ]:
spark.stop()